<div style='text-align: center; padding: 30px'>
  <h1><strong>Create the Optimum Portfolios</strong></h1>
  <h3><strong>Heriberto Espino Montelongo</strong></h1>
</div>

In [1]:
import pandas as pd
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns

(CVXPY) Mar 23 02:33:17 AM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Mar 23 02:33:17 AM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')


In [ ]:
file = r'cryptos.csv'
data = pd.read_csv(file, index_col="Date", parse_dates=True)

In [20]:
mu = expected_returns.mean_historical_return(data)
s = risk_models.sample_cov(data)

# **MAX SHARPE BUY ONLY**

In [21]:
efbuy = EfficientFrontier(mu, s)    # Efficient Frontier Buy

In [22]:
efbuycopy = efbuy.deepcopy()

In [23]:
efbuyms = efbuy.max_sharpe()        # Efficient Frontier Buy Max Sharpe

In [24]:
wefbuy = efbuy.clean_weights()      # Weights Efficient Frontier Buy

In [25]:
for i in wefbuy:
    if wefbuy[i] > 0:
        print(i, wefbuy[i])

XRP 0.05507
TRX 0.21625
LEO 0.72868


In [26]:
efbuy.portfolio_performance(verbose=True)

Expected annual return: 25.2%
Annual volatility: 39.4%
Sharpe Ratio: 0.59


(0.25193168177684233, 0.3936818390910312, 0.5891348260116532)

# **MAX SHARPE CFDs**

In [27]:
efcfd = EfficientFrontier(mu, s, weight_bounds=(-1, 1))    # Efficient Frontier Buy

In [28]:
import cvxpy as cp

# all the sums of the abs(weights) must be less than or equal to 1, add the constraint
efcfd.add_constraint(lambda x: cp.sum(cp.abs(x)) <= 1)


In [29]:
efcfdcopy = efcfd.deepcopy()        # a copy for not calculating again

In [30]:
efcfdms = efcfd.max_sharpe()        # Efficient Frontier Buy Max Sharpe

c:\Users\herie\AppData\Local\Programs\Python\Python312\Lib\site-packages\cvxpy\problems\problem.py:1407: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


In [31]:
wefcfd = efcfd.clean_weights()      # Weights Efficient Frontier Buy

In [32]:
sum = 0
for crypto in wefcfd:
    if wefcfd[crypto] != 0:
        print(crypto, wefcfd[crypto])
        sum = sum + abs(wefcfd[crypto])

XRP 0.05489
TRX 0.21621
TON -0.00027
LEO 0.72896


In [16]:
print(sum)

1.0003099999999998


In [17]:
efcfd.portfolio_performance(verbose=True)

Expected annual return: 25.2%
Annual volatility: 39.4%
Sharpe Ratio: 0.59


(0.2521955395554794, 0.3938325938580408, 0.589579286165369)

In [18]:
# save the weights to a csv file
pd.DataFrame(wefbuy, index=[0]).to_csv('weights_buy.csv', index=False)
pd.DataFrame(wefcfd, index=[0]).to_csv('weights_cfd.csv', index=False)